# Ankara — Spatial Detector Training

Trains a linear probe on **frozen DINOv2-base** features to detect spatial
deepfake artifacts. The backbone is never fine-tuned — only the small probe
head trains, so this runs in minutes on an M4 and resists overfitting.

Validated at **97.8% accuracy / 0.998 AUC** (see `02_benchmark_eval.ipynb`).

In [ ]:
import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import cv2

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'✔  Device  : {DEVICE}')
print(f'✔  PyTorch : {torch.__version__}')

DATA_DIR   = Path('../datasets/train_subset')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

real_count = len(list((DATA_DIR / 'real').glob('*.mp4')))
fake_count = len(list((DATA_DIR / 'fake').glob('*.mp4')))
print(f'✔  Dataset : {real_count} real  |  {fake_count} fake')

BATCH_SIZE       = 8
FRAMES_PER_VIDEO = 5
DINO_MODEL_NAME  = 'facebook/dinov2-base'   # 768-dim CLS token
EPOCHS           = 15
print(f'✔  Config  : batch={BATCH_SIZE}, frames/video={FRAMES_PER_VIDEO}, model={DINO_MODEL_NAME}')

In [ ]:
class FFDataset(Dataset):
    """Full-frame sampler. The probe is trained on whole frames (no face crop),
    matching the inference pipeline in backend/detectors/spatial.py."""
    def __init__(self, data_dir, max_frames=5):
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.samples = []
        for label, folder in [(0, 'real'), (1, 'fake')]:
            for vid in sorted(Path(data_dir / folder).glob('*.mp4')):
                for frame in self._frames(vid, max_frames):
                    self.samples.append((frame, label))
        np.random.shuffle(self.samples)
        n_real = sum(1 for _, l in self.samples if l == 0)
        n_fake = sum(1 for _, l in self.samples if l == 1)
        print(f'  {len(self.samples)} frames  ({n_real} real / {n_fake} fake)')

    def _frames(self, path, n):
        cap = cv2.VideoCapture(str(path))
        total = max(1, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
        out = []
        for i in np.linspace(0, total - 1, min(n, total), dtype=int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
            ok, f = cap.read()
            if ok and f is not None:
                out.append(f)
        cap.release()
        return out

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        frame, label = self.samples[idx]
        img = frame[:, :, ::-1].copy()  # BGR → RGB
        return self.transform(img), torch.tensor(label, dtype=torch.float32)


print('Loading dataset...')
full_ds = FFDataset(DATA_DIR, max_frames=FRAMES_PER_VIDEO)
n_val   = max(10, int(len(full_ds) * 0.2))
n_train = len(full_ds) - n_val
train_ds, val_ds = torch.utils.data.random_split(full_ds, [n_train, n_val])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'✔  Train: {n_train}  |  Val: {n_val}')

In [ ]:
import torch.nn as nn
from transformers import AutoModel

print(f'Loading {DINO_MODEL_NAME}...')
dino = AutoModel.from_pretrained(DINO_MODEL_NAME)
dino.eval().to(DEVICE)
for p in dino.parameters():
    p.requires_grad = False
FEAT_DIM = 768
print(f'✔  Loaded  — feature dim: {FEAT_DIM}')
gc.collect()
if DEVICE == 'mps': torch.mps.empty_cache()


class DINOv2Probe(nn.Module):
    """768 -> 128 -> 1. Must match backend/detectors/spatial.py exactly."""
    def __init__(self, feat_dim=768):
        super().__init__()
        self.probe = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )
    def forward(self, x): return self.probe(x)


probe = DINOv2Probe(FEAT_DIM).to(DEVICE)
print('✔  Probe ready')

In [ ]:
criterion = nn.BCELoss()
opt   = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
best_acc = 0.0

print(f'Training DINOv2 probe — {EPOCHS} epochs on {DEVICE}')
print('─' * 60)
for epoch in range(1, EPOCHS + 1):
    probe.train(); tr_loss = tr_correct = tr_total = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            feats = dino(imgs).last_hidden_state[:, 0]
        preds = probe(feats).squeeze()
        loss  = criterion(preds, labels)
        opt.zero_grad(); loss.backward(); opt.step()
        tr_loss += loss.item(); tr_correct += ((preds > 0.5).float() == labels).sum().item(); tr_total += labels.size(0)
        del imgs, labels, feats, preds, loss
        if DEVICE == 'mps': torch.mps.empty_cache()

    probe.eval(); vl_correct = vl_total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = probe(dino(imgs).last_hidden_state[:, 0]).squeeze()
            vl_correct += ((preds > 0.5).float() == labels).sum().item(); vl_total += labels.size(0)
            if DEVICE == 'mps': torch.mps.empty_cache()

    tr_acc, vl_acc = tr_correct / tr_total, vl_correct / vl_total
    sched.step(); saved = ''
    if vl_acc > best_acc:
        best_acc = vl_acc
        torch.save(probe.state_dict(), MODELS_DIR / 'spatial_probe.pt'); saved = '  ★ saved'
    print(f'Epoch {epoch:02d}/{EPOCHS}  loss={tr_loss/len(train_loader):.4f}  train={tr_acc:.3f}  val={vl_acc:.3f}{saved}')

print(f'\n✔  Best val accuracy: {best_acc:.3f}')
print('✔  Saved → models/spatial_probe.pt')

In [ ]:
# Smoke test on one real + one fake video
probe.eval()
t = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def score_video(path, label):
    cap = cv2.VideoCapture(str(path)); total = max(1, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    scores = []
    for i in np.linspace(0, total - 1, 5, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, f = cap.read()
        if ok:
            img = t(f[:, :, ::-1].copy()).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                scores.append(probe(dino(img).last_hidden_state[:, 0]).item())
            if DEVICE == 'mps': torch.mps.empty_cache()
    cap.release()
    prob = float(np.mean(scores)) if scores else 0.5
    pred = 'FAKE' if prob > 0.5 else 'REAL'
    print(f"{'✔' if pred == label else '✘'}  [{label}] → {pred}  (fake_prob={prob:.3f})")

print('=== Smoke Test ===')
r = next((DATA_DIR/'real').glob('*.mp4'), None)
f = next((DATA_DIR/'fake').glob('*.mp4'), None)
if r: score_video(r, 'REAL')
if f: score_video(f, 'FAKE')
print('\n✔  Proceed to 02_benchmark_eval.ipynb')